In [2]:
# AutoML - Agriculture Crop Health Application
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [3]:
# 1. Create agriculture dataset
np.random.seed(42)
n = 500
data = pd.DataFrame({
    "Soil_Moisture": np.random.randint(20, 81, n),
    "Temperature": np.random.randint(15, 41, n),
    "Humidity": np.random.randint(30, 91, n),
    "Soil_pH": np.random.uniform(4.5, 8.5, n).round(1),
    "Rainfall_mm": np.random.randint(0, 301, n),
    "Fertilizer_kg": np.random.randint(0, 201, n),
    "Sunlight_Hours": np.random.uniform(3, 12, n).round(1),
    "Pest_Level": np.random.randint(0, 11, n)
})
# Create meaningful target score
score = (
    data["Soil_Moisture"] * 0.20
    + data["Humidity"] * 0.10
    + data["Rainfall_mm"] * 0.08
    + data["Fertilizer_kg"] * 0.05
    + data["Sunlight_Hours"] * 3
    - abs(data["Temperature"] - 27) * 3
    - abs(data["Soil_pH"] - 6.5) * 12
    - data["Pest_Level"] * 8
)
# Add randomness
score = score + np.random.normal(0, 15, n)
# 1 = Healthy Crop, 0 = Needs Attention
data["Crop_Health"] = np.where(
    score >= np.median(score),
    1,
    0
)
print("Agriculture dataset created successfully!")
print("Shape:", data.shape)
display(data.head())


Agriculture dataset created successfully!
Shape: (500, 9)


,Soil_Moisture,Temperature,Humidity,Soil_pH,Rainfall_mm,Fertilizer_kg,Sunlight_Hours,Pest_Level,Crop_Health
0,58,40,63,6.7,23,20,7.8,9,0
1,71,17,35,7.9,292,41,7.5,3,0
2,48,32,82,6.6,3,140,3.9,7,0
3,34,39,31,5.2,219,112,8.9,5,0
4,62,38,42,5.7,19,104,10.4,3,1


In [4]:
# 2. Separate features and target
X = data.drop("Crop_Health", axis=1)
y = data["Crop_Health"]
print("\nFeatures:")
print(X.columns.tolist())
print("\nTarget:")
print("Crop_Health")


Features:
['Soil_Moisture', 'Temperature', 'Humidity', 'Soil_pH', 'Rainfall_mm', 'Fertilizer_kg', 'Sunlight_Hours', 'Pest_Level']

Target:
Crop_Health


In [5]:
# 3. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)
print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 375
Testing samples: 125


In [6]:
# 4. Define models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "Decision Tree": Pipeline([
        ("model", DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(random_state=42))
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ])
}

In [7]:
# 5. Hyperparameter grids
param_grids = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9],
        "model__weights": ["uniform", "distance"]
    },
    "Decision Tree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_split": [2, 5, 10]
    },
    "Random Forest": {
        "model__n_estimators": [50, 100, 150],
        "model__max_depth": [3, 5, 8, None]
    },
    "SVM": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"]
    }
}

In [8]:
# 6. Train and evaluate models
results = []
best_model = None
best_model_name = None
best_accuracy = 0
best_parameters = None
for model_name, model in models.items():
    print("\n" + "=" * 70)
    print("Training:", model_name)
    print("=" * 70)
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    tuned_model = grid.best_estimator_
    y_pred = tuned_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    print("Best Parameters:", grid.best_params_)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Best Parameters": grid.best_params_
    })
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = tuned_model
        best_model_name = model_name
        best_parameters = grid.best_params_


Training: Logistic Regression
Best Parameters: {'model__C': 0.1}
Accuracy : 0.7840
Precision: 0.7778
Recall   : 0.7903
F1 Score : 0.7840

Training: KNN
Best Parameters: {'model__n_neighbors': 9, 'model__weights': 'uniform'}
Accuracy : 0.7680
Precision: 0.7391
Recall   : 0.8226
F1 Score : 0.7786

Training: Decision Tree
Best Parameters: {'model__max_depth': 2, 'model__min_samples_split': 2}
Accuracy : 0.7760
Precision: 0.7179
Recall   : 0.9032
F1 Score : 0.8000

Training: Random Forest
Best Parameters: {'model__max_depth': None, 'model__n_estimators': 50}
Accuracy : 0.8240
Precision: 0.8333
Recall   : 0.8065
F1 Score : 0.8197

Training: SVM
Best Parameters: {'model__C': 0.1, 'model__kernel': 'rbf'}
Accuracy : 0.7840
Precision: 0.7692
Recall   : 0.8065
F1 Score : 0.7874


In [9]:
# 7. Compare models
results_df = pd.DataFrame(results).sort_values(
    by="Accuracy",
    ascending=False
)
print("\n" + "=" * 80)
print("AUTOML AGRICULTURE MODEL COMPARISON")
print("=" * 80)
display(results_df)


AUTOML AGRICULTURE MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
3,Random Forest,0.824,0.833333,0.806452,0.819672,"{'model__max_depth': None, 'model__n_estimator..."
0,Logistic Regression,0.784,0.777778,0.790323,0.784000,{'model__C': 0.1}
4,SVM,0.784,0.769231,0.806452,0.787402,"{'model__C': 0.1, 'model__kernel': 'rbf'}"
2,Decision Tree,0.776,0.717949,0.903226,0.800000,"{'model__max_depth': 2, 'model__min_samples_sp..."
1,KNN,0.768,0.739130,0.822581,0.778626,"{'model__n_neighbors': 9, 'model__weights': 'u..."


In [10]:
# 8. Best model
print("\n" + "=" * 70)
print("BEST MODEL")
print("=" * 70)
print("Model:", best_model_name)
print("Accuracy:", round(best_accuracy, 4))
print("Best Parameters:", best_parameters)


BEST MODEL
Model: Random Forest
Accuracy: 0.824
Best Parameters: {'model__max_depth': None, 'model__n_estimators': 50}


In [11]:
# 9. Classification report and confusion matrix
best_predictions = best_model.predict(X_test)
print("\nClassification Report")
print("=" * 70)
print(
    classification_report(
        y_test,
        best_predictions,
        target_names=["Needs Attention", "Healthy"]
    )
)
cm = confusion_matrix(y_test, best_predictions)
print("Confusion Matrix:")
print(cm)


Classification Report
                 precision    recall  f1-score   support

Needs Attention       0.82      0.84      0.83        63
        Healthy       0.83      0.81      0.82        62

       accuracy                           0.82       125
      macro avg       0.82      0.82      0.82       125
   weighted avg       0.82      0.82      0.82       125

Confusion Matrix:
[[53 10]
 [12 50]]


In [12]:
# 10. Save best model
joblib.dump(
    best_model,
    "best_crop_health_model.pkl"
)
print("\nBest model saved successfully!")
print("File: best_crop_health_model.pkl")


Best model saved successfully!
File: best_crop_health_model.pkl


In [ ]:
# 11. Load model and predict crop sample 1
loaded_model = joblib.load("best_crop_health_model.pkl")
crop_sample_1 = pd.DataFrame({
    "Soil_Moisture": [65],
    "Temperature": [27],
    "Humidity": [70],
    "Soil_pH": [6.5],
    "Rainfall_mm": [180],
    "Fertilizer_kg": [100],
    "Sunlight_Hours": [8.0],
    "Pest_Level": [2]
})
prediction_1 = loaded_model.predict(crop_sample_1)
if prediction_1[0] == 1:
    result_1 = "HEALTHY"
else:
    result_1 = "NEEDS ATTENTION"

print("\nCrop Sample 1 Prediction:", result_1)


Crop Sample 1 Prediction: HEALTHY


In [ ]:
# 12. Predict crop sample 2
crop_sample_2 = pd.DataFrame({
    "Soil_Moisture": [25],
    "Temperature": [38],
    "Humidity": [35],
    "Soil_pH": [8.2],
    "Rainfall_mm": [20],
    "Fertilizer_kg": [15],
    "Sunlight_Hours": [4.0],
    "Pest_Level": [9]
})
prediction_2 = loaded_model.predict(crop_sample_2)
if prediction_2[0] == 1:
    result_2 = "HEALTHY"
else:
    result_2 = "NEEDS ATTENTION"
print("Crop Sample 2 Prediction:", result_2)

Crop Sample 2 Prediction: NEEDS ATTENTION
